<a href="https://colab.research.google.com/github/JoeBosi/fuel-token-demo/blob/main/notebooks/fuel_price_oracle_it.ipynb" target="_blank" rel="noopener"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Apri in Colab"></a>

# Token Benzina: leggi il prezzo e confrontalo con la blockchain

Esercizio del tutorial di Token Benzina. In cinque celle: leggi il prezzo della benzina dal servizio pubblico del Ministero (Osservaprezzi), calcoli la **mediana**, leggi lo stesso prezzo dal contratto **oracolo** sulla rete di prova Polygon Amoy e confronti i due numeri. Non servono chiavi, portafogli o account: premi **Esegui tutto** (menu Runtime). In Colab serve un account Google gratuito per eseguire le celle.

**Cella 1.** Installa le due librerie (circa un minuto).

In [1]:
!pip install -q web3 requests

**Cella 2.** Importa gli strumenti.

In [2]:
import requests
from statistics import median
from web3 import Web3

**Cella 3.** Chiama il servizio Osservaprezzi con un `User-Agent` chiaro, un timeout e messaggi di errore comprensibili.

In [3]:
# 1) Read petrol prices around Arezzo from the Italian "Osservaprezzi" service.
# A clear User-Agent and Accept header keep public services happy (many answer HTTP 403 without them).
URL = "https://carburanti.mise.gov.it/ospzApi/search/zone"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; educational-demo)",
    "Accept": "application/json",
    "Content-Type": "application/json",
}
body = {"points": [{"lat": 43.4636, "lng": 11.8748}], "radius": 30, "fuelType": "1-x", "priceOrder": "asc"}

resp = requests.post(URL, json=body, headers=HEADERS, timeout=30)
if resp.status_code == 403:
    raise SystemExit("HTTP 403: the service refused the request, check the User-Agent header")
if resp.status_code == 429:
    raise SystemExit("HTTP 429: too many requests, wait a minute and run the cell again")
resp.raise_for_status()
data = resp.json()
print("HTTP", resp.status_code, "- stations received:", len(data["results"]))

HTTP 200 - stations received: 44


**Cella 4.** Tiene i prezzi self-service della benzina e calcola la mediana.

In [4]:
# 2) Keep only self-service unleaded petrol (fuelId 1) and take the median.
prices = [
    float(f["price"])
    for station in data["results"]
    for f in station["fuels"]
    if f["fuelId"] == 1 and f["isSelf"] is True and float(f["price"]) > 0
]
offchain = median(prices)
print(f"Median petrol price: {offchain:.3f} EUR/L over {len(prices)} stations")

Median petrol price: 1.990 EUR/L over 42 stations


**Cella 5.** Legge il prezzo dal contratto oracolo su Amoy (sola lettura). Il contratto salva i prezzi con 6 decimali.

In [5]:
# 3) Read the same price from the oracle contract on Polygon Amoy (read only, no wallet, no key).
RPC = "https://polygon-amoy-bor-rpc.publicnode.com"
ORACLE = "0xf30D07696476e97103730BE0dFb55045945409D8"  # fuel/contracts/deployments/amoy.json
ABI = [
    {"type": "function", "name": "fuelPrice", "stateMutability": "view", "inputs": [],
     "outputs": [{"name": "price", "type": "uint256"}, {"name": "updatedAt", "type": "uint64"}]},
    {"type": "function", "name": "isFresh", "stateMutability": "view",
     "inputs": [{"name": "maxAgeSeconds", "type": "uint64"}], "outputs": [{"type": "bool"}]},
]
w3 = Web3(Web3.HTTPProvider(RPC, request_kwargs={"timeout": 30}))
print("Connected, chain id:", w3.eth.chain_id)  # 80002 = Polygon Amoy

oracle = w3.eth.contract(address=Web3.to_checksum_address(ORACLE), abi=ABI)
raw, updated_at = oracle.functions.fuelPrice().call()
onchain = raw / 1_000_000  # the contract stores 6 decimals: 1990000 means 1.99 EUR
print(f"On-chain petrol price: {onchain} EUR/L (raw value {raw})")
print("Price is fresh (3 days):", oracle.functions.isFresh(3 * 86400).call())

Connected, chain id: 80002


On-chain petrol price: 1.99 EUR/L (raw value 1990000)
Price is fresh (3 days): True


**Cella 6.** Confronta prezzo del servizio e prezzo sulla catena. L'oracolo si aggiorna una volta al giorno, quindi una piccola differenza è normale.

In [6]:
# 4) Compare the two numbers. The oracle is updated once a day, so a small gap is normal.
gap = abs(offchain - onchain)
print(f"Off-chain {offchain:.3f} vs on-chain {onchain:.3f} -> gap {gap:.3f} EUR/L")
print("Close enough (under 5 cents):", gap < 0.05)

Off-chain 1.990 vs on-chain 1.990 -> gap 0.000 EUR/L
Close enough (under 5 cents): True


Fatto. Torna al tutorial: https://rwa-fuel.blockchain.giuseppebosi.com/tutorial/it/ch7.html